# Práctica 1: Limpieza de Datos

**Dataset:** Spotify Data (obtenido de [HuggingFace](https://huggingface.co/datasets/khepplewhite/SpotifyData), descargado con `data/raw/dataset.py`)


In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", 40)

PATH_RAW = "../data/raw/spotify_raw.csv"
PATH_PROCESSED = "../data/processed/spotify_cleaned.csv"


## 1. Carga de datos crudos

In [2]:
df_raw = pd.read_csv(PATH_RAW)
print(f"Dimensiones iniciales: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas.")
df_raw.head()

Dimensiones iniciales: 66174 filas y 36 columnas.


,Unnamed: 0,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,artist_img,collab,track_name,release_date,album_num_tracks,album_cover,source,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language,pivot
0,29477,spotify:track:4ZtFanR9U6ndgddUvNcjcG,1,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,good 4 u,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,1,1,3,3612666,03-06-2021,0.563,0.664,9,1,-5.044,0.1540,0.335,0.000000,0.0849,0.6880,166.928,178147,Australia,Oceania,English,0
1,29478,spotify:track:6HU7h9RYOaPRFeh0R3UeAr,3,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,deja vu,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,2,2,9,1666474,03-06-2021,0.442,0.612,2,1,-7.222,0.1120,0.584,0.000006,0.3700,0.1780,180.917,215507,Australia,Oceania,English,0
2,29479,spotify:track:5CZ40GBx1sQ9agT82CLQCT,4,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,traitor,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,4,6,2,1433290,03-06-2021,0.380,0.339,3,1,-7.885,0.0338,0.691,0.000000,0.1200,0.0849,100.607,229227,Australia,Oceania,English,0
3,29480,spotify:track:5wANPM4fQCJwkGd4rN57mH,6,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,drivers license,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,1,5,21,1266266,03-06-2021,0.561,0.431,10,1,-8.810,0.0578,0.768,0.000014,0.1060,0.1370,143.875,242013,Australia,Oceania,English,0
4,29481,spotify:track:5JCoSi02qi3jJeHdZXMmR8,7,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,favorite crime,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,7,15,2,1184068,03-06-2021,0.369,0.272,9,1,-10.497,0.0364,0.866,0.000000,0.1470,0.2180,172.929,152667,Australia,Oceania,English,0


## 2. Diagnóstico inicial

Antes de limpiar, se revisan: tipos de dato por columna, valores nulos y filas duplicadas.

In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 66174 entries, 0 to 66173
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         66174 non-null  int64  
 1   uri                66174 non-null  str    
 2   rank               66174 non-null  int64  
 3   artist_names       66174 non-null  str    
 4   artists_num        66174 non-null  int64  
 5   artist_individual  66174 non-null  str    
 6   artist_id          66174 non-null  str    
 7   artist_genre       66174 non-null  str    
 8   artist_img         65974 non-null  str    
 9   collab             66174 non-null  int64  
 10  track_name         66174 non-null  str    
 11  release_date       66174 non-null  str    
 12  album_num_tracks   66174 non-null  int64  
 13  album_cover        66174 non-null  str    
 14  source             66174 non-null  str    
 15  peak_rank          66174 non-null  int64  
 16  previous_rank      66174 non-null

In [4]:
nulos = df_raw.isnull().sum()
print("Valores nulos por columna:")
nulos[nulos > 0] if nulos.sum() > 0 else print("No se encontraron valores nulos.")

Valores nulos por columna:


artist_img    200
dtype: int64

In [5]:
duplicados = df_raw.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

Filas duplicadas: 0


## 3. Limpieza de datos

Se aplican, en orden, los siguientes pasos:

1. **Eliminación de duplicados exactos.**
2. **Imputación de valores nulos**: columnas categóricas → `"Unknown"`, columnas numéricas → mediana de la columna.
3. **Estandarización de nombres de columnas** a `snake_case`.
4. **Validación de tipos**: se descartan filas con `duration <= 0` (duración inválida).


### 3.1 Eliminación de duplicados exactos

In [6]:
df_clean = df_raw.copy()

filas_antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Se eliminaron {filas_antes - len(df_clean)} filas duplicadas.")
print(f"Filas restantes: {len(df_clean)}")

Se eliminaron 0 filas duplicadas.
Filas restantes: 66174


### 3.2 Imputación de valores nulos

In [7]:
cols_texto = df_clean.select_dtypes(include=["object", "string", "str"]).columns
for col in cols_texto:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna("Unknown")
        print(f"Imputados valores nulos en columna categórica: '{col}' con 'Unknown'.")

cols_num = df_clean.select_dtypes(include=["number"]).columns
for col in cols_num:
    if df_clean[col].isnull().sum() > 0:
        mediana = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(mediana)
        print(f"Imputados valores nulos en columna numérica: '{col}' con la mediana ({mediana}).")

if df_clean.isnull().sum().sum() == 0:
    print("No había valores nulos que imputar (el dataset ya llegaba completo).")

Imputados valores nulos en columna categórica: 'artist_img' con 'Unknown'.


No había valores nulos que imputar (el dataset ya llegaba completo).


### 3.3 Estandarización de nombres de columnas

In [8]:
columnas_antes = df_clean.columns.tolist()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("[^a-zA-Z0-2_]", "", regex=True)
)

print("Nombres de columnas estandarizados (snake_case).")
pd.DataFrame({"antes": columnas_antes, "después": df_clean.columns.tolist()})

Nombres de columnas estandarizados (snake_case).


,antes,después
0,Unnamed: 0,unnamed_0
1,uri,uri
2,rank,rank
3,artist_names,artist_names
4,artists_num,artists_num
5,artist_individual,artist_individual
6,artist_id,artist_id
7,artist_genre,artist_genre
8,artist_img,artist_img
9,collab,collab


### 3.4 Validación de tipos (duración positiva)

In [9]:
if "duration" in df_clean.columns:
    filas_antes = len(df_clean)
    df_clean = df_clean[df_clean["duration"] > 0]
    print(f"Se eliminaron {filas_antes - len(df_clean)} filas con duration <= 0.")

print(f"\nDimensiones finales tras limpieza: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas.")
df_clean.head()

Se eliminaron 0 filas con duration <= 0.

Dimensiones finales tras limpieza: 66174 filas y 36 columnas.


,unnamed_0,uri,rank,artist_names,artists_num,artist_individual,artist_id,artist_genre,artist_img,collab,track_name,release_date,album_num_tracks,album_cover,source,peak_rank,previous_rank,weeks_on_chart,streams,week,danceability,energy,key,mode,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,country,region,language,pivot
0,29477,spotify:track:4ZtFanR9U6ndgddUvNcjcG,1,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,good 4 u,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,1,1,3,3612666,03-06-2021,0.563,0.664,9,1,-5.044,0.1540,0.335,0.000000,0.0849,0.6880,166.928,178147,Australia,Oceania,English,0
1,29478,spotify:track:6HU7h9RYOaPRFeh0R3UeAr,3,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,deja vu,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,2,2,9,1666474,03-06-2021,0.442,0.612,2,1,-7.222,0.1120,0.584,0.000006,0.3700,0.1780,180.917,215507,Australia,Oceania,English,0
2,29479,spotify:track:5CZ40GBx1sQ9agT82CLQCT,4,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,traitor,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,4,6,2,1433290,03-06-2021,0.380,0.339,3,1,-7.885,0.0338,0.691,0.000000,0.1200,0.0849,100.607,229227,Australia,Oceania,English,0
3,29480,spotify:track:5wANPM4fQCJwkGd4rN57mH,6,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,drivers license,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,1,5,21,1266266,03-06-2021,0.561,0.431,10,1,-8.810,0.0578,0.768,0.000014,0.1060,0.1370,143.875,242013,Australia,Oceania,English,0
4,29481,spotify:track:5JCoSi02qi3jJeHdZXMmR8,7,Olivia Rodrigo,1,Olivia Rodrigo,spotify:artist:1McMsnEElThX1knmY4oliG,pop,https://i.scdn.co/image/ab6761610000e5ebee954a...,0,favorite crime,21-05-2021,11,https://i.scdn.co/image/ab67616d0000b273a91c10...,Olivia Rodrigo PS,7,15,2,1184068,03-06-2021,0.369,0.272,9,1,-10.497,0.0364,0.866,0.000000,0.1470,0.2180,172.929,152667,Australia,Oceania,English,0


## 4. Verificación final

Se comprueba que ya no queden valores nulos ni duplicados

In [10]:
print("Nulos restantes:", df_clean.isnull().sum().sum())
print("Duplicados restantes:", df_clean.duplicated().sum())
df_clean.dtypes

Nulos restantes: 0


Duplicados restantes: 0


unnamed_0              int64
uri                      str
rank                   int64
artist_names             str
artists_num            int64
artist_individual        str
artist_id                str
artist_genre             str
artist_img               str
collab                 int64
track_name               str
release_date             str
album_num_tracks       int64
album_cover              str
source                   str
peak_rank              int64
previous_rank          int64
weeks_on_chart         int64
streams                int64
week                     str
danceability         float64
energy               float64
key                    int64
mode                   int64
loudness             float64
speechiness          float64
acousticness         float64
instrumentalness     float64
liveness             float64
valence              float64
tempo                float64
duration               int64
country                  str
region                   str
language      

## 5. Guardado del dataset limpio

El dataset limpio se guarda en `data/processed/spotify_cleaned.csv`, listo para usarse en las prácticas siguientes.

In [11]:
os.makedirs(os.path.dirname(PATH_PROCESSED), exist_ok=True)
df_clean.to_csv(PATH_PROCESSED, index=False, encoding="utf-8")
print(f"Dataset limpio guardado exitosamente en: {PATH_PROCESSED}")

Dataset limpio guardado exitosamente en: ../data/processed/spotify_cleaned.csv
